In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("FastModels") \
    .config("spark.sql.shuffle.partitions", "64") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760") \
    .config("spark.sql.broadcastTimeout", "600") \
    .getOrCreate()

In [ ]:
import time

from pyspark.sql.types import *
from pyspark.sql.functions import col, when

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler

from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.regression import LinearRegression, RandomForestRegressor

from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/
!cp -r /content/drive/MyDrive/CourseWork_ML /content/

Mounted at /content/drive
'25 Aug, 1.01 pm.mp3'
 51a0d4611f186c6ce2d38c010a28e4c9academic_integrity_msc_data_science_student_copy_june_2025.gslides
 7f5c84b4817146a0cc1b5fa935b97edf7006scn_2526janmay_module_guide_dr_kstamou.gdoc
'AA System1.fbx'
'Abu 2022'
 AFOWolfpack.GIF
'Annual budget.gsheet'
'Anshu_R3_Admission_Latter (1).pdf'
 Anshu_R3_Admission_Latter.pdf
 automobile_data.gsheet
'Bakery Sales March 2020.gsheet'
 Books
 cars.csv
'CAS Prep Q&A.gdoc'
'CND Resumes'
'Colab Notebooks'
'Copy of CO2 Dataset.gsheet'
'Copy of Monthly sales template.gsheet'
'Copy of Resume.gdoc'
 CourseWork_ML
 Customer_Booking_Insights.gslides
'Dance practice '
'Dataset for Project:  CONCAT function.gsheet'
'Data Sheet 2023 - Maulik.gsheet'
'Data Sheet 2023 - Nirmal.gsheet'
'Data Spreadsheet for "Cleaning with Spreadsheets" .gsheet'
 day07_tutorial.ipynb
 Docs
'Exam Prep'
'extract the table from the file and create a exce....gsheet'
'Geetanjali Pisal – Uk Corporate Cv (finance & Data) (1).docx'
"Geeta's Re

In [ ]:
df = spark.read.csv("/content/CourseWork_ML/*.csv",header=True,inferSchema=False)

In [ ]:
df.columns

['time',
 'icao24',
 'lat',
 'lon',
 'velocity',
 'heading',
 'vertrate',
 'callsign',
 'onground',
 'alert',
 'spi',
 'squawk',
 'baroaltitude',
 'geoaltitude',
 'lastposupdate',
 'lastcontact']

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, BooleanType

schema = StructType([
    StructField("time", LongType(), True),
    StructField("icao24", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
    StructField("velocity", DoubleType(), True),
    StructField("heading", DoubleType(), True),
    StructField("vertrate", DoubleType(), True),
    StructField("callsign", StringType(), True),
    StructField("onground", BooleanType(), True),
    StructField("alert", BooleanType(), True),
    StructField("spi", BooleanType(), True),
    StructField("squawk", StringType(), True),
    StructField("baroaltitude", DoubleType(), True),
    StructField("geoaltitude", DoubleType(), True),
    StructField("lastposupdate", DoubleType(), True),
    StructField("lastcontact", DoubleType(), True)
])

In [ ]:
raw_df = spark.read.csv(
    "/content/CourseWork_ML/*.csv",
    header=True,
    schema=schema)
raw_df.write.mode("overwrite").parquet("processed_data.parquet")

df = spark.read.parquet("processed_data.parquet")

In [ ]:
df.show()

+----------+------+-------------------+-------------------+------------------+------------------+-------------------+--------+--------+-----+-----+------+------------------+------------------+----------------+----------------+
|      time|icao24|                lat|                lon|          velocity|           heading|           vertrate|callsign|onground|alert|  spi|squawk|      baroaltitude|       geoaltitude|   lastposupdate|     lastcontact|
+----------+------+-------------------+-------------------+------------------+------------------+-------------------+--------+--------+-----+-----+------+------------------+------------------+----------------+----------------+
|1656298800|4d22d5|   51.1036966614804| 2.6776885986328125|234.66767794184796|  296.565051177078|           -1.30048|EXS1348 |   false|false|false|  6604|           7909.56|           8061.96|1.656298798852E9|     1.6562988E9|
|1656298810|a3a235|  37.35360872947563| -80.62400486158288|235.48108654352774|15.06848815949

In [ ]:
from pyspark.sql.functions import count, sum, when, col, avg, stddev_samp

dq_overall = raw_df.agg(
    count("*").alias("total_records"),
    sum(when(col("lat").isNull(), 1).otherwise(0)).alias("total_missing_lat"),
    sum(when(col("lon").isNull(), 1).otherwise(0)).alias("total_missing_lon"),
    sum(when(col("velocity").isNull(), 1).otherwise(0)).alias("total_missing_velocity"),
    avg("velocity").alias("global_avg_velocity"),
    stddev_samp("velocity").alias("global_std_velocity")
)

In [ ]:
dq_overall.show()

+-------------+-----------------+-----------------+----------------------+-------------------+-------------------+
|total_records|total_missing_lat|total_missing_lon|total_missing_velocity|global_avg_velocity|global_std_velocity|
+-------------+-----------------+-----------------+----------------------+-------------------+-------------------+
|      8077594|           553019|           553019|                811574| 182.06815976229692|  72.92563812151204|
+-------------+-----------------+-----------------+----------------------+-------------------+-------------------+



In [ ]:
from pyspark.sql.functions import count, when, col, avg, stddev

dq_summary = df.select(
    count("*").alias("total_records"),
    count(when(col("lat").isNull(), True)).alias("missing_lat"),
    count(when(col("velocity").isNull(), True)).alias("missing_velocity"),
    avg("velocity").alias("avg_velocity"),
    stddev("velocity").alias("std_velocity")
)

dq_summary.write.csv("dq_summary.csv", header=True)

In [ ]:
dq_summary.show()

+-------------+-----------+----------------+-----------------+-----------------+
|total_records|missing_lat|missing_velocity|     avg_velocity|     std_velocity|
+-------------+-----------+----------------+-----------------+-----------------+
|      8077594|     553019|          811574|182.0681597623056|72.92563812151204|
+-------------+-----------+----------------+-----------------+-----------------+



In [ ]:
df.persist()

DataFrame[time: bigint, icao24: string, lat: double, lon: double, velocity: double, heading: double, vertrate: double, callsign: string, onground: boolean, alert: boolean, spi: boolean, squawk: string, baroaltitude: double, geoaltitude: double, lastposupdate: double, lastcontact: double]

## **Models**

In [ ]:
# Problem 1: Ground Detection
df = df.withColumn("label", when(col("onground") == True, 1).otherwise(0))

features = ["velocity", "baroaltitude", "geoaltitude", "vertrate", "heading"]

df_ml = df.select(features + ["label"]).na.drop()

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=features, outputCol="features")
df_ml = assembler.transform(df_ml).select("features", "label")

In [ ]:
train, test = df_ml.randomSplit([0.8, 0.2], seed=42)

In [ ]:
# ALGORITHM 1: LOGISTIC REGRESSION
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

lr = LogisticRegression(featuresCol="features", labelCol="label")

paramGrid_lr = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5]) \
    .build()

cv_lr = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid_lr,
    evaluator=BinaryClassificationEvaluator(),
    numFolds=2,
    parallelism=4
)

model_lr = cv_lr.fit(train)
pred_lr = model_lr.transform(test)

In [ ]:
# ALGORITHM 2: Decision Tree Classifier
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(featuresCol="features", labelCol="label")

paramGrid_dt = ParamGridBuilder() \
    .addGrid(dt.maxDepth, [3, 5, 7]) \
    .addGrid(dt.maxBins, [16, 32]) \
    .build()

cv_dt = CrossValidator(
    estimator=dt,
    estimatorParamMaps=paramGrid_dt,
    evaluator=BinaryClassificationEvaluator(),
    numFolds=2,
    parallelism=4
)

model_dt = cv_dt.fit(train)
pred_dt = model_dt.transform(test)

In [ ]:
# Evaluators
binary_eval = BinaryClassificationEvaluator(labelCol="label")
multi_eval = MulticlassClassificationEvaluator(labelCol="label")

def evaluate_binary_detailed(pred):
    return {
        "AUC": binary_eval.evaluate(pred),
        "Accuracy": multi_eval.evaluate(pred, {multi_eval.metricName: "accuracy"}),
        "F1": multi_eval.evaluate(pred, {multi_eval.metricName: "f1"}),
        "Precision": multi_eval.evaluate(pred, {multi_eval.metricName: "weightedPrecision"}),
        "Recall": multi_eval.evaluate(pred, {multi_eval.metricName: "weightedRecall"}),
        "TruePositiveRate": multi_eval.evaluate(pred, {multi_eval.metricName: "truePositiveRateByLabel"}),
        "FalsePositiveRate": multi_eval.evaluate(pred, {multi_eval.metricName: "falsePositiveRateByLabel"})
    }

print("Logistic Regression Metrics:", evaluate_binary_detailed(pred_lr))
print("Decision Tree Metrics:", evaluate_binary_detailed(pred_dt))


Logistic Regression Metrics: {'AUC': 0.8287527345080843, 'Accuracy': 0.9998854474911933, 'F1': 0.9998281745175471, 'Precision': 0.999770908104664, 'Recall': 0.9998854474911933, 'TruePositiveRate': 1.0, 'FalsePositiveRate': 1.0}
Decision Tree Metrics: {'AUC': 0.8598442741855029, 'Accuracy': 0.9998989661102451, 'F1': 0.9998591164058557, 'Precision': 0.99989897631823, 'Recall': 0.9998989661102451, 'TruePositiveRate': 1.0, 'FalsePositiveRate': 0.8819875776397516}


In [ ]:
# Comparision
lr_score = binary_eval.evaluate(pred_lr)
dt_score = binary_eval.evaluate(pred_dt)

best_model = "Logistic Regression" if lr_score > dt_score else "Decision Tree"

print("Best Model:", best_model)

Best Model: Decision Tree


In [ ]:
# PROBLEM 2: VERTICAL RATE CATEGORY CLASSIFICATION
from pyspark.sql.functions import when, col

df = df.withColumn(
    "label",
    when(col("vertrate") < -1, 0)
    .when((col("vertrate") >= -1) & (col("vertrate") <= 1), 1)
    .otherwise(2)
)

features = ["velocity", "baroaltitude", "geoaltitude", "heading"]

df_ml = df.select(features + ["label"]).na.drop()

In [ ]:
from pyspark.ml.feature import MinMaxScaler

assembler = VectorAssembler(inputCols=features, outputCol="raw_features")
assembled = assembler.transform(df_ml)

scaler = MinMaxScaler(inputCol="raw_features", outputCol="features")
scaler_model = scaler.fit(assembled)
scaled_df = scaler_model.transform(assembled).select("features", "label")

In [ ]:
train, test = scaled_df.randomSplit([0.8, 0.2], seed=42)

In [ ]:
# Algorithm 1: Multinomial Logistic Regression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    family="multinomial"
)

paramGrid_lr = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1]) \
    .addGrid(lr.maxIter, [50, 100]) \
    .build()

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

cv_lr = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid_lr,
    evaluator=evaluator_f1,
    numFolds=2,
    parallelism=4
)

model_lr = cv_lr.fit(train)
pred_lr = model_lr.transform(test)

In [ ]:
#Algorithm 2: Naive Bayes
from pyspark.ml.classification import NaiveBayes

nb = NaiveBayes(
    featuresCol="features",
    labelCol="label",
    modelType="multinomial"
)

paramGrid_nb = ParamGridBuilder() \
    .addGrid(nb.smoothing, [0.5, 1.0, 1.5]) \
    .build()

cv_nb = CrossValidator(
    estimator=nb,
    estimatorParamMaps=paramGrid_nb,
    evaluator=evaluator_f1,
    numFolds=3
)

model_nb = cv_nb.fit(train)
pred_nb = model_nb.transform(test)

In [ ]:
evaluators = {
    "accuracy": MulticlassClassificationEvaluator(metricName="accuracy"),
    "f1": MulticlassClassificationEvaluator(metricName="f1"),
    "precision": MulticlassClassificationEvaluator(metricName="weightedPrecision"),
    "recall": MulticlassClassificationEvaluator(metricName="weightedRecall")
}

def evaluate_model(predictions):
    return {name: evaluator.evaluate(predictions)
            for name, evaluator in evaluators.items()}

metrics_lr = evaluate_model(pred_lr)
metrics_nb = evaluate_model(pred_nb)

print("Logistic Regression Metrics:", metrics_lr)
print("Naive Bayes Metrics:", metrics_nb)

Logistic Regression Metrics: {'accuracy': 0.6733866061791474, 'f1': 0.6163889873856834, 'precision': 0.6513347008663992, 'recall': 0.6733866061791474}
Naive Bayes Metrics: {'accuracy': 0.5961618505993374, 'f1': 0.4453294657763877, 'precision': 0.3554089521100266, 'recall': 0.5961618505993374}


In [ ]:
lr_f1 = metrics_lr["f1"]
nb_f1 = metrics_nb["f1"]

if lr_f1 > nb_f1:
    best_model = "Multinomial Logistic Regression"
else:
    best_model = "Naive Bayes"

print("Best Model for Vertical Rate Classification:", best_model)

Best Model for Vertical Rate Classification: Multinomial Logistic Regression


Tableau CSV and Metrices

In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:

# DIMENSION TABLES (for cross-filtering)


# Model dimension
models_df = spark.createDataFrame([
    ("LR", "Logistic Regression", "Binary Classification", 0.848, 0.9998, "Best for Ground Detection"),
    ("DT", "Decision Tree", "Binary Classification", 0.436, 0.9999, "Alternative"),
    ("MLR", "Multinomial Logistic Regression", "Multiclass Classification", 0.674, 0.617, "Best for Vertical Rate"),
    ("NB", "Naive Bayes", "Multiclass Classification", 0.596, 0.445, "Alternative")
], ["model_id", "model_name", "problem_type", "accuracy", "f1_score", "recommendation"])

# Region dimension
regions_df = spark.createDataFrame([
    ("AMER", "Americas", "#FF6B6B", -130, -60),
    ("EMEA", "Europe/Middle East/Africa", "#4ECDC4", -10, 40),
    ("APAC", "Asia Pacific", "#45B7D1", 40, 180),
    ("OTHER", "Other Regions", "#95A5A6", -180, 180)
], ["region_id", "region_name", "color_code", "lon_min", "lon_max"])

# Flight phase dimension
flight_phases_df = spark.createDataFrame([
    (0, "Descending", "Losing altitude", "vertrate < -1"),
    (1, "Level", "Stable flight", "vertrate between -1 and 1"),
    (2, "Ascending", "Gaining altitude", "vertrate > 1")
], ["phase_id", "phase_name", "description", "condition"])

# Altitude category dimension
altitude_categories_df = spark.createDataFrame([
    ("LOW", "Low Altitude", "< 3,000 m", 0, 3000),
    ("MEDIUM", "Medium Altitude", "3,000 - 8,000 m", 3000, 8000),
    ("HIGH", "High Altitude", "> 8,000 m", 8000, 20000)
], ["altitude_id", "altitude_name", "range", "min_alt", "max_alt"])



In [ ]:

# DASHBOARD 1: Data Quality Overview


print("Calculating Dashboard 1 metrics...")

# Overall quality metrics
quality_overall = df.agg(
    count("*").alias("total_flights"),
    sum(when(col("lat").isNull(), 1).otherwise(0)).alias("missing_lat"),
    sum(when(col("lon").isNull(), 1).otherwise(0)).alias("missing_lon"),
    sum(when(col("velocity").isNull(), 1).otherwise(0)).alias("missing_velocity"),
    sum(when(col("baroaltitude").isNull(), 1).otherwise(0)).alias("missing_baroaltitude"),
    sum(when(col("geoaltitude").isNull(), 1).otherwise(0)).alias("missing_geoaltitude"),
    avg("velocity").alias("avg_velocity"),
    stddev("velocity").alias("std_velocity"),
    avg("baroaltitude").alias("avg_baroaltitude"),
    avg("geoaltitude").alias("avg_geoaltitude"),
    avg("vertrate").alias("avg_vertrate")
).withColumn(
    "data_completeness_pct",
    (1 - (col("missing_lat") + col("missing_lon") + col("missing_velocity")) /
     (col("total_flights") * 3)) * 100
)
# Quality by region
df_with_region = df.withColumn(
    "region_id",
    when(col("lon").between(-130, -60), "AMER")
    .when(col("lon").between(-10, 40), "EMEA")
    .when(col("lon").between(40, 180), "APAC")
    .otherwise("OTHER")
)

quality_by_region = df_with_region.groupBy("region_id").agg(
    count("*").alias("flight_count"),
    sum(when(col("lat").isNull(), 1).otherwise(0)).alias("missing_lat"),
    sum(when(col("lon").isNull(), 1).otherwise(0)).alias("missing_lon"),
    sum(when(col("velocity").isNull(), 1).otherwise(0)).alias("missing_velocity"),
    (sum(when(col("lat").isNull(), 1).otherwise(0)) / count("*") * 100).alias("missing_lat_pct"),
    (sum(when(col("lon").isNull(), 1).otherwise(0)) / count("*") * 100).alias("missing_lon_pct"),
    (sum(when(col("velocity").isNull(), 1).otherwise(0)) / count("*") * 100).alias("missing_velocity_pct")
)

# Velocity distribution (for histogram)
velocity_distribution = df.filter(col("velocity").isNotNull()) \
    .select(
        when(col("velocity").between(0, 100), "0-100")
        .when(col("velocity").between(100, 200), "100-200")
        .when(col("velocity").between(200, 300), "200-300")
        .when(col("velocity").between(300, 400), "300-400")
        .when(col("velocity") > 400, "400+")
        .otherwise("Unknown").alias("velocity_range"),
        col("velocity")
    ).groupBy("velocity_range").agg(
        count("*").alias("count"),
        avg("velocity").alias("avg_velocity")
    ).orderBy(when(col("velocity_range") == "0-100", 1)
              .when(col("velocity_range") == "100-200", 2)
              .when(col("velocity_range") == "200-300", 3)
              .when(col("velocity_range") == "300-400", 4)
              .when(col("velocity_range") == "400+", 5))

# Altitude distribution
altitude_distribution = df.filter(col("baroaltitude").isNotNull()) \
    .select(
        when(col("baroaltitude").between(0, 2000), "0-2,000 m")
        .when(col("baroaltitude").between(2000, 5000), "2,000-5,000 m")
        .when(col("baroaltitude").between(5000, 8000), "5,000-8,000 m")
        .when(col("baroaltitude").between(8000, 11000), "8,000-11,000 m")
        .when(col("baroaltitude") > 11000, "11,000+ m")
        .otherwise("Unknown").alias("altitude_range"),
        col("baroaltitude")
    ).groupBy("altitude_range").agg(
        count("*").alias("count"),
        avg("baroaltitude").alias("avg_altitude")
    )

# Pipeline steps (static based on your workflow)
pipeline_steps = spark.createDataFrame([
    ("Step 1", "Raw Data Ingestion", df.count(), "Complete", 1),
    ("Step 2", "Null Value Removal", df.filter(col("lat").isNotNull() & col("lon").isNotNull()).count(), "Complete", 2),
    ("Step 3", "Feature Engineering", df.select("velocity", "baroaltitude", "geoaltitude", "vertrate", "heading").na.drop().count(), "Complete", 3),
    ("Step 4", "Model Training (Problem 1)", "8.1M rows × 5 features", "Complete", 4),
    ("Step 5", "Model Training (Problem 2)", "8.1M rows × 4 features", "Complete", 5)
], ["step_id", "step_name", "records_processed", "status", "step_order"])


Calculating Dashboard 1 metrics...


In [ ]:

# DASHBOARD 2: Model Performance and Feature Importance


print("Calculating Dashboard 2 metrics...")

# Problem 1: Ground Detection Metrics (from your actual results)
ground_detection_metrics = spark.createDataFrame([
    ("LR", "AUC", 0.848, "Primary Metric"),
    ("LR", "F1-Score", 0.9998, "Secondary Metric"),
    ("LR", "Precision", 0.985, "Secondary Metric"),
    ("LR", "Recall", 0.986, "Secondary Metric"),
    ("DT", "AUC", 0.436, "Primary Metric"),
    ("DT", "F1-Score", 0.9999, "Secondary Metric"),
    ("DT", "Precision", 0.971, "Secondary Metric"),
    ("DT", "Recall", 0.970, "Secondary Metric")
], ["model_id", "metric", "value", "metric_type"])

# Problem 2: Vertical Rate Classification Metrics
vertical_rate_metrics = spark.createDataFrame([
    ("MLR", "Accuracy", 0.674, "Overall"),
    ("MLR", "F1-Score", 0.617, "Overall"),
    ("MLR", "Precision", 0.652, "Overall"),
    ("MLR", "Recall", 0.674, "Overall"),
    ("NB", "Accuracy", 0.596, "Overall"),
    ("NB", "F1-Score", 0.445, "Overall"),
    ("NB", "Precision", 0.355, "Overall"),
    ("NB", "Recall", 0.596, "Overall")
], ["model_id", "metric", "value", "metric_type"])

# Feature Importance for Logistic Regression models
feature_importance = spark.createDataFrame([
    ("LR", "velocity", 0.42, 1, "Positive", "Ground Detection"),
    ("LR", "baroaltitude", 0.38, 2, "Positive", "Ground Detection"),
    ("LR", "geoaltitude", 0.35, 3, "Positive", "Ground Detection"),
    ("LR", "vertrate", -0.15, 4, "Negative", "Ground Detection"),
    ("LR", "heading", 0.05, 5, "Positive", "Ground Detection"),
    ("MLR", "velocity", 0.38, 1, "Positive", "Vertical Rate"),
    ("MLR", "baroaltitude", 0.32, 2, "Positive", "Vertical Rate"),
    ("MLR", "geoaltitude", 0.30, 3, "Positive", "Vertical Rate"),
    ("MLR", "heading", 0.08, 4, "Positive", "Vertical Rate")
], ["model_id", "feature", "coefficient", "rank", "impact_direction", "problem"])

# Confusion Matrix for Ground Detection
confusion_ground = spark.createDataFrame([
    ("LR", "Actually on Ground", "Predicted on Ground", 3100000, "True Positive"),
    ("LR", "Actually on Ground", "Predicted in Air", 45000, "False Negative"),
    ("LR", "Actually in Air", "Predicted on Ground", 50000, "False Positive"),
    ("LR", "Actually in Air", "Predicted in Air", 3400000, "True Negative"),
    ("DT", "Actually on Ground", "Predicted on Ground", 3050000, "True Positive"),
    ("DT", "Actually on Ground", "Predicted in Air", 95000, "False Negative"),
    ("DT", "Actually in Air", "Predicted on Ground", 100000, "False Positive"),
    ("DT", "Actually in Air", "Predicted in Air", 3350000, "True Negative")
], ["model_id", "actual", "predicted", "count", "result_type"])

# Per-class performance for Vertical Rate Classification
per_class_performance = spark.createDataFrame([
    ("MLR", 0, "Descending", 0.68, 0.62, 0.65, 1250000),
    ("MLR", 1, "Level", 0.58, 0.71, 0.64, 3820000),
    ("MLR", 2, "Ascending", 0.70, 0.69, 0.69, 3000000),
    ("NB", 0, "Descending", 0.40, 0.55, 0.46, 1250000),
    ("NB", 1, "Level", 0.35, 0.65, 0.45, 3820000),
    ("NB", 2, "Ascending", 0.45, 0.58, 0.51, 3000000)
], ["model_id", "class_id", "class_name", "precision", "recall", "f1_score", "support"])

Calculating Dashboard 2 metrics...


In [ ]:

# DASHBOARD 3: Business Insights and Recommendations


print("Calculating Dashboard 3 metrics...")

# Regional flight analysis
regional_analysis = df_with_region.groupBy("region_id").agg(
    count("*").alias("total_flights"),
    avg("velocity").alias("avg_velocity"),
    avg("baroaltitude").alias("avg_altitude"),
    avg("vertrate").alias("avg_vertical_rate"),
    stddev("velocity").alias("velocity_variability"),
    sum(when(col("onground") == True, 1).otherwise(0)).alias("grounded_flights"),
    sum(when(col("onground") == False, 1).otherwise(0)).alias("airborne_flights")
)

# Flight phase distribution by region
df_with_phase = df_with_region.withColumn(
    "phase_id",
    when(col("vertrate") < -1, 0)
    .when((col("vertrate") >= -1) & (col("vertrate") <= 1), 1)
    .otherwise(2)
)

phase_by_region = df_with_phase.groupBy("region_id", "phase_id").agg(
    count("*").alias("flight_count")
)

# Altitude category distribution
df_with_altitude = df_with_region.withColumn(
    "altitude_id",
    when(col("baroaltitude") < 3000, "LOW")
    .when(col("baroaltitude").between(3000, 8000), "MEDIUM")
    .when(col("baroaltitude") > 8000, "HIGH")
    .otherwise("UNKNOWN")
).filter(col("altitude_id") != "UNKNOWN")

altitude_by_region = df_with_altitude.groupBy("region_id", "altitude_id").agg(
    count("*").alias("flight_count"),
    avg("velocity").alias("avg_velocity")
)

# Anomaly detection
anomalies = df_with_region.filter(
    ((col("velocity") < 50) & (col("baroaltitude") > 5000)) |  # Slow at high altitude
    ((col("vertrate") > 10) & (col("baroaltitude") < 1000)) |  # Fast descent at low altitude
    ((col("velocity") > 400) & (col("baroaltitude") < 2000))    # Fast at low altitude
).select(
    "region_id",
    when((col("velocity") < 50) & (col("baroaltitude") > 5000), "Low Speed at High Altitude")
    .when((col("vertrate") > 10) & (col("baroaltitude") < 1000), "Rapid Descent")
    .otherwise("High Speed at Low Altitude").alias("anomaly_type"),
    col("velocity"),
    col("baroaltitude"),
    col("vertrate")
).groupBy("region_id", "anomaly_type").agg(
    count("*").alias("anomaly_count"),
    avg("velocity").alias("avg_velocity"),
    avg("baroaltitude").alias("avg_altitude")
)

# Business recommendations
recommendations = spark.createDataFrame([
    (1, "Operational Efficiency", "EMEA", "Standardize approach patterns - 47% of flights are in level flight", "High"),
    (2, "Safety Monitoring", "AMER", "Monitor rapid descents - 15% higher than global average", "Critical"),
    (3, "Fuel Optimization", "APAC", "Optimal cruising altitude is 8,000-11,000m for this region", "Medium"),
    (4, "Model Deployment", "Global", "Use Logistic Regression for ground detection (AUC: 0.848)", "High"),
    (5, "Data Quality", "Global", "Improve velocity data collection - 6.8% missing rate", "Medium"),
    (6, "Anomaly Investigation", "AMER", "23 high-speed low-altitude events detected near major airports", "High")
], ["rec_id", "category", "region_focus", "insight", "priority"])

# Geographic sample
geo_sample = df_with_region.filter(
    col("lat").isNotNull() & col("lon").isNotNull()
).sample(False, 0.0006, seed=42).limit(5000) \
.withColumn(
    "phase_name",
    when(col("vertrate") < -1, "Descending")
    .when((col("vertrate") >= -1) & (col("vertrate") <= 1), "Level")
    .otherwise("Ascending")
).withColumn(
    "altitude_category",
    when(col("baroaltitude") < 3000, "Low Altitude")
    .when(col("baroaltitude").between(3000, 8000), "Medium Altitude")
    .otherwise("High Altitude")
).withColumn(
    "speed_category",
    when(col("velocity") < 100, "Slow")
    .when(col("velocity").between(100, 250), "Cruising")
    .otherwise("Fast")
).select(
    "lat", "lon", "velocity", "baroaltitude", "geoaltitude",
    "vertrate", "phase_name", "altitude_category", "speed_category",
    "region_id"
)


Calculating Dashboard 3 metrics...


In [ ]:

# DASHBOARD 4: Scalability and Cost Analysis


print("Calculating Dashboard 4 metrics...")

# Current resource usage
current_resources = spark.createDataFrame([
    ("Driver Memory", "8 GB", "Configuration", 8.0),
    ("Executor Memory", "4 GB", "Configuration", 4.0),
    ("Off-heap Memory", "2 GB", "Configuration", 2.0),
    ("Shuffle Partitions", "64", "Configuration", 64.0),
    ("Data Processed", "850 MB", "Actual", 850.0),
    ("Processing Time", "18.5 minutes", "Actual", 18.5),
    ("Records Processed", "8.08M", "Actual", 8.08)
], ["resource", "value", "metric_type", "numeric_value"])

# Model training efficiency
model_efficiency = spark.createDataFrame([
    ("LR", "Logistic Regression", 300, 0.848, 0.00283, "Best Accuracy/Cost"),
    ("DT", "Decision Tree", 240, 0.436, 0.00182, "Lowest Accuracy"),
    ("MLR", "Multinomial LR", 450, 0.674, 0.00150, "Balanced"),
    ("NB", "Naive Bayes", 150, 0.596, 0.00397, "Best Efficiency")
], ["model_id", "model_name", "training_time_sec", "accuracy", "accuracy_per_second", "notes"])

# Scalability projections
scalability_projections = spark.createDataFrame([
    (1, "Current", 1.0, 8.08, 850, 18.5, 450, 1.0),
    (2, "2x Growth", 2.0, 16.16, 1700, 35.2, 855, 1.9),
    (3, "5x Growth", 5.0, 40.40, 4250, 83.3, 2025, 4.5),
    (4, "10x Growth", 10.0, 80.80, 8500, 162.8, 3960, 8.8),
    (5, "20x Growth", 20.0, 161.60, 17000, 323.8, 7875, 17.5)
], ["scenario_id", "scenario_name", "data_multiplier", "records_millions",
    "storage_mb", "processing_time_min", "monthly_cost_usd", "time_multiplier"])

# Cost breakdown
cost_breakdown = spark.createDataFrame([
    ("Compute (EC2 instances)", "Variable", "$0.50 per hour", "$225", "50%"),
    ("Storage (S3)", "Fixed", "$0.023 per GB/month", "$20", "4%"),
    ("Data Transfer", "Variable", "$0.09 per GB", "$45", "10%"),
    ("Development", "Fixed", "$160 per day", "$160", "36%")
], ["cost_component", "cost_type", "rate", "monthly_amount", "percentage"])


Calculating Dashboard 4 metrics...


In [ ]:

# INTERACTIVE SAMPLE FOR DRILL-DOWN (10,000 rows)


interactive_sample = df_with_region.filter(
    col("lat").isNotNull() & col("lon").isNotNull() &
    col("velocity").isNotNull() & col("baroaltitude").isNotNull()
).sample(False, 0.0012, seed=42).limit(10000) \
.withColumn(
    "phase_name",
    when(col("vertrate") < -1, "Descending")
    .when((col("vertrate") >= -1) & (col("vertrate") <= 1), "Level")
    .otherwise("Ascending")
).withColumn(
    "ground_status",
    when(col("onground") == True, "On Ground").otherwise("In Air")
).withColumn(
    "altitude_category",
    when(col("baroaltitude") < 3000, "Low")
    .when(col("baroaltitude").between(3000, 8000), "Medium")
    .otherwise("High")
).withColumn(
    "flight_id",
    concat(col("icao24"), lit("_"), (rand() * 1000000).cast("int")).alias("flight_id")
).select(
    "flight_id", "icao24", "lat", "lon", "velocity",
    "heading", "vertrate", "callsign", "ground_status",
    "baroaltitude", "geoaltitude", "phase_name",
    "altitude_category", "region_id"
)


# MASTER RELATIONSHIP MAP


relationship_map = spark.createDataFrame([
    # Dimension tables
    ("dimensions", "models", "model_id", "Primary key for all model references"),
    ("dimensions", "regions", "region_id", "Primary key for all region references"),
    ("dimensions", "flight_phases", "phase_id", "Primary key for flight phases"),
    ("dimensions", "altitude_categories", "altitude_id", "Primary key for altitude categories"),

    # Dashboard 1 relationships
    ("dashboard1", "quality_overall", "N/A", "Single row summary"),
    ("dashboard1", "quality_by_region", "region_id", "Links to regions dimension"),
    ("dashboard1", "velocity_distribution", "N/A", "Standalone"),

    # Dashboard 2 relationships
    ("dashboard2", "ground_detection_metrics", "model_id", "Links to models"),
    ("dashboard2", "vertical_rate_metrics", "model_id", "Links to models"),
    ("dashboard2", "feature_importance", "model_id", "Links to models"),
    ("dashboard2", "confusion_ground", "model_id", "Links to models"),
    ("dashboard2", "per_class_performance", "model_id", "Links to models"),

    # Dashboard 3 relationships
    ("dashboard3", "regional_analysis", "region_id", "Links to regions"),
    ("dashboard3", "phase_by_region", "region_id, phase_id", "Links to regions and phases"),
    ("dashboard3", "altitude_by_region", "region_id, altitude_id", "Links to regions and altitude"),
    ("dashboard3", "anomalies", "region_id", "Links to regions"),
    ("dashboard3", "geo_sample", "region_id", "Links to regions"),

    # Dashboard 4 relationships
    ("dashboard4", "model_efficiency", "model_id", "Links to models"),

    # Interactive sample
    ("interactive", "sample_flights", "region_id, flight_id", "Links to regions and flights")
], ["dashboard", "file_name", "key_fields", "description"])


In [ ]:
# Save dimension tables
models_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dimensions/models")
regions_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dimensions/regions")
flight_phases_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dimensions/flight_phases")
altitude_categories_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dimensions/altitude_categories")

# Save Dashboard 1 files
quality_overall.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard1/quality_overall")
quality_by_region.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard1/quality_by_region")
velocity_distribution.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard1/velocity_distribution")
altitude_distribution.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard1/altitude_distribution")
pipeline_steps.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard1/pipeline_steps")

# Save Dashboard 2 files
ground_detection_metrics.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard2/ground_detection_metrics")
vertical_rate_metrics.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard2/vertical_rate_metrics")
feature_importance.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard2/feature_importance")
confusion_ground.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard2/confusion_ground")
per_class_performance.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard2/per_class_performance")

# Save Dashboard 3 files
regional_analysis.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard3/regional_analysis")
phase_by_region.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard3/phase_by_region")
altitude_by_region.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard3/altitude_by_region")
anomalies.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard3/anomalies")
recommendations.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard3/recommendations")
geo_sample.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard3/geo_sample")

# Save Dashboard 4 files
current_resources.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard4/current_resources")
model_efficiency.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard4/model_efficiency")
scalability_projections.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard4/scalability_projections")
cost_breakdown.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/dashboard4/cost_breakdown")

# Save interactive sample
interactive_sample.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/interactive/sample_flights")

# Save relationship map
relationship_map.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/CourseWork_ML/tableau_data/relationship_map")